In [ ]:
# Installation of ifcopenshell
!pip install ifcopenshell

In [ ]:
# Resets the coordinate system of an ifc file
# Usecase: a file was exported wrong, like georeferenced but needs to be in local coordinates for coordination
# Version: 0.1 - 07.07.2025 - IFC4x3

import ifcopenshell
import math
import os # Import os module to manipulate file paths

# Input fields for coordinates and angle
input_x = 0.0  # Default x-coordinate
input_y = 0.0  # Default y-coordinate
input_z = 461500.0  # Default z-coordinate
input_angle = 0.0 # Default angle in degrees

# Loading the ifc file
input_filename = 'LU32_LI_MD_ME_FACHM.ifc'
ifc_file = ifcopenshell.open(input_filename)

# Find the IfcSite entity
site = ifc_file.by_type("IfcSite")

site_geometric_info_file1 = {}

# Check if an IfcSite was found and select the first one if it's a list
if site:
    if isinstance(site, list):
        site = site[0]

    # Access the ObjectPlacement
    object_placement = site.ObjectPlacement

    if object_placement:
        # Access the RelativePlacement (should be IfcAxis2Placement3D)
        relative_placement = object_placement.RelativePlacement

        if relative_placement and relative_placement.is_a("IfcAxis2Placement3D"):
            # Extract Location (IfcCartesianPoint)
            location = relative_placement.Location
            if location and location.is_a("IfcCartesianPoint"):
                site_geometric_info_file1["Location"] = location.Coordinates
            else:
                site_geometric_info_file1["Location"] = "Not found or invalid type"

            # Extract RefDirection (IfcDirection)
            ref_direction = relative_placement.RefDirection
            if ref_direction and ref_direction.is_a("IfcDirection"):
                site_geometric_info_file1["RefDirection"] = ref_direction.DirectionRatios

                # Calculate angle from RefDirection if it exists
                if site_geometric_info_file1["RefDirection"] is not None and len(site_geometric_info_file1["RefDirection"]) >= 2:
                    try:
                        # Assuming the direction is in the XY plane for a simple angle calculation
                        dx = site_geometric_info_file1["RefDirection"][0]
                        dy = site_geometric_info_file1["RefDirection"][1]
                        angle_radians = math.atan2(dy, dx)
                        angle_degrees = math.degrees(angle_radians)
                        site_geometric_info_file1["RefDirection_Angle_Degrees"] = angle_degrees
                    except Exception as e:
                        site_geometric_info_file1["RefDirection_Angle_Degrees"] = f"Could not calculate angle: {e}"
                else:
                     site_geometric_info_file1["RefDirection_Angle_Degrees"] = "Insufficient data for angle calculation"

            else:
                site_geometric_info_file1["RefDirection"] = "Not found or invalid type"
                site_geometric_info_file1["RefDirection_Angle_Degrees"] = "RefDirection not found or invalid type"


            # Extract Axis (IfcDirection)
            axis = relative_placement.Axis
            if axis and axis.is_a("IfcDirection"):
                site_geometric_info_file1["Axis"] = axis.DirectionRatios
                 # Calculate angle from Axis if it exists (assuming in XY plane for simplicity)
                if site_geometric_info_file1["Axis"] is not None and len(site_geometric_info_file1["Axis"]) >= 2:
                    try:
                        dx = site_geometric_info_file1["Axis"][0]
                        dy = site_geometric_info_file1["Axis"][1]
                        angle_radians = math.atan2(dy, dx)
                        angle_degrees = math.degrees(angle_radians)
                        site_geometric_info_file1["Axis_Angle_Degrees"] = angle_degrees
                    except Exception as e:
                        site_geometric_info_file1["Axis_Angle_Degrees"] = f"Could not calculate angle: {e}"
                else:
                    site_geometric_info_file1["Axis_Angle_Degrees"] = "Insufficient data for angle calculation"
            else:
                site_geometric_info_file1["Axis"] = "Not found or invalid type"
                site_geometric_info_file1["Axis_Angle_Degrees"] = "Axis not found or invalid type"

            print("Extracted Original IfcSite geometric information:")
            display(site_geometric_info_file1)

            # --- Start of modification code ---

            # 1. Create new entities
            # Create a new IfcCartesianPoint with the input coordinates
            new_location = ifc_file.createIfcCartesianPoint((input_x, input_y, input_z))

            # Calculate direction ratios from the input angle (assuming XY plane)
            angle_radians = math.radians(input_angle)
            new_dx = math.cos(angle_radians)
            new_dy = math.sin(angle_radians)
            # Assuming z-component remains 0 for the RefDirection in XY plane
            new_direction_ratios = (new_dx, new_dy, 0.0)

            # Create a new IfcDirection with the calculated direction ratios
            new_ref_direction = ifc_file.createIfcDirection(new_direction_ratios)

            # 2. Replace existing entities
            # Assign the new Location and RefDirection to the RelativePlacement
            relative_placement.Location = new_location
            relative_placement.RefDirection = new_ref_direction

            # 3. Update and print information
            # Update the site_geometric_info_file1 dictionary with the new values
            site_geometric_info_file1["Location"] = new_location.Coordinates
            site_geometric_info_file1["RefDirection"] = new_ref_direction.DirectionRatios
            site_geometric_info_file1["RefDirection_Angle_Degrees"] = input_angle # Use input angle directly

            # --- End of modification code ---


        else:
            site_geometric_info_file1["RelativePlacement"] = "Not found or invalid type"
    else:
        site_geometric_info_file1["ObjectPlacement"] = "Not found"

    print("\nModified IfcSite geometric information:")
    display(site_geometric_info_file1)

else:
    print("No IfcSite entity found in the first file.")
    site_geometric_info_file1 = {} # Initialize as empty if no site found


# --- Start of Unit Extraction Code ---
units = ifc_file.by_type("IfcUnitAssignment")
if units:
    # Assuming the first IfcUnitAssignment contains the relevant units
    unit_assignment = units[0]
    for unit in unit_assignment.Units:
        if unit.is_a("IfcSIUnit"):
            # Check for length units
            if unit.UnitType == "LENGTHUNIT":
                print(f"\nLength Unit: {unit.Name}")
                # You can further check unit.Prefix for things like MILLI
                if unit.Prefix:
                    print(f"Prefix: {unit.Prefix}")
else:
    print("\nNo IfcUnitAssignment found in the file.")
# --- End of Unit Extraction Code ---


# Construct the output filename
base_name, ext = os.path.splitext(input_filename)
output_filename = f"{base_name}_modified{ext}"

# Save the modified IFC file
ifc_file.write(output_filename)
print(f"\nIFC created with modified coordinates: {output_filename}")

Extracted Original IfcSite geometric information:


{'Location': (2665235965.2999997, 1212210805.6999996, 461500.0),
 'RefDirection': (0.8716157333343193, 0.490189772847289, 0.0),
 'RefDirection_Angle_Degrees': 29.35305555555552,
 'Axis': (0.0, 0.0, 1.0),
 'Axis_Angle_Degrees': 0.0}


Modified IfcSite geometric information:


{'Location': (0.0, 0.0, 461500.0),
 'RefDirection': (1.0, 0.0, 0.0),
 'RefDirection_Angle_Degrees': 0.0,
 'Axis': (0.0, 0.0, 1.0),
 'Axis_Angle_Degrees': 0.0}


Length Unit: METRE
Prefix: MILLI

IFC created with modified coordinates: LU32_LI_MD_ME_FACHM_modified.ifc
